## Simulación de sistema enrutador

In [ ]:
import os
import json
from dotenv import load_dotenv
from ollama import Client
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

In [ ]:
openai_api_key = os.getenv('OLLAMA_HOST')
groq_api_key = os.getenv('GROQ_API_KEY')

In [ ]:
if openai_api_key:
    print(f"La clave API de OpenAI existe y empieza por {openai_api_key[:8]}")
else:
    print("La clave API de OpenAI no existe.")

if groq_api_key:
    print(f"la clave API de Groq existe y empieza por {groq_api_key[:8]}")

else:
    print("La clave API de Groq no existe")

In [ ]:
import re


models = ["gpt-oss:120b-cloud", "kimi-k2-thinking:cloud", "deepseek-v3.1:671b-cloud", "kimi-k2.5:cloud"]
request = f"Eres un agente especializado en reparto de tareas y tienes que elegir entre los siguientes modelos elegiendo el modelo más óptimo según la tarea que se te pida: {models}: "
request += f"El modelo {models[0]} tiene un I/O cost de $0.35/2.0 , con un SPEED de 260 t/s y un Context size de 131,072; " 
request += f"El modelo {models[1]} tiene un I/O cost de $0.6/2.5 con un SPEED de 79 t/s y un Context size de 256,000; "
request += f"El modelo {models[2]} tiene un I/O cost de $0.27/1.1 con un SPEED de 330 t/s y un Context size de 128,000."
request += f"El modelo {models[3]} tiene un I/O cost de $0.05/0.10 con un SPEED de 160 t/s y un Context size de 70,000."
#request += f"Tienes que responder SOLO con el nombre del modelo que has elegido, no tienes que explicar nada, solo decir el nombre del modelo ya que tu finción es elegir el más óptimo para la tarea que se te pide"
#request += f"**Responde con un array de dos posiciones donde la posicion 0 será el modelo elegido y la posicion 1 el razonamiento del porqué has elegido ese modelo**; **Muy importante que devuelvas un array de dos posiciones como te he indicado porque es el formato que espera el siguiente LLM**"
request += f""" **Responde con formato JSON y solo JSON con el siguiente formato: {{"modelo":"modelo elegido de los 3 posibles", "razonamiento": "Explicación con razonamiento del porqué el modelo elegido y no otro"}}** """
request += f" La tarea a realizar es: "
tarea = "¿Pueden los LLMs soñar con ovejas eléctricas?" #Tarea a realizar por el LLM
messages = [{"role": "user", "content": request+tarea}]
print(messages)

## LLM Router

In [ ]:
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "llama-3.3-70b-versatile"

response = groq.chat.completions.create(model=model_name, messages=messages)
formatoJson = response.choices[0].message.content

results_dict = json.loads(formatoJson)

modelo = results_dict["modelo"]
razonamiento = results_dict["razonamiento"]

print(f"El modelo elegido es: {modelo}")
print(f"El razonamiento ha sido: {razonamiento}")


## LLM ejecutor 

In [ ]:
ollama = Client()

messages = [{"role": "user", "content": tarea}]

response = ollama.chat(
    model = modelo,
    messages = messages,
)
answer = response.message.content

display(Markdown(answer))